![](../img/logo_ucm.jpg)

# Ejercicio resuelto: Airflow Training Pipeline

Esta version muestra una solucion de referencia para el DAG didáctico de entrenamiento. Se mantiene el mismo alcance funcional del enunciado, pero sobre un `dag_id` distinto, `ml_pipeline_preprocessing_training_exercise_resuelta`, para no interferir con el DAG real del proyecto.

In [ ]:
from pathlib import Path
import json
import textwrap
import yaml


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "airflow_project").exists() and (candidate / "notebooks_clase").exists():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto")


PROJECT_ROOT = find_project_root()
AIRFLOW_PROJECT = PROJECT_ROOT / "airflow_project"
SOLVED_DAG_PATH = AIRFLOW_PROJECT / "dags" / "airflow_ml_pipeline_resuelta.py"
CONFIG_PATH = AIRFLOW_PROJECT / "config" / "config.yaml"
VARIABLES_PATH = AIRFLOW_PROJECT / "config" / "variables.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DAG resuelto:", SOLVED_DAG_PATH)

## Objetivos

1. Generar una implementación completa de ejercicio de entrenamiento con Airflow y sus DAGs.
2. Entender como se interconectan `ReaderFactory`, `TrainerFactory`, MLflow y el registro condicional.
3. Tener una referencia con un stack de Docker y Docker Compose. 

In [ ]:
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
variables = json.loads(VARIABLES_PATH.read_text(encoding="utf-8"))

print("EXPERIMENT_NAME =", config["mlflow"]["experiment_name"])
print("MODEL_NAME =", config["mlflow"]["model_name"])
print("Variables por defecto =", variables)

## Apartado 1: comandos de preparación

```bash
cd airflow_project
docker compose up --build
```

URLs de trabajo:

- Airflow: `http://127.0.0.1:8080`
- MLflow: `http://127.0.0.1:5000`

DAG resuelto disponible en Airflow:

- `ml_pipeline_preprocessing_training_exercise_resuelta`

## Apartado 2: solución de ReaderFactory y TrainerFactory

La solucion registra `local` y `delta` en `ReaderFactory`, aunque el segundo se deja fuera de alcance evaluable. `TrainerFactory` soporta tres clasificadores: `random_forest`, `logistic_regression` y `xgboost`.

In [ ]:
solved_dag_text = SOLVED_DAG_PATH.read_text(encoding="utf-8")
for marker in ["class ReaderFactory", "class TrainerFactory"]:
    index = solved_dag_text.index(marker)
    print("=" * 100)
    print(solved_dag_text[index:index + 900])
    print()

## Apartado 3: entrenamiento y logging en MLflow

La tarea `train_and_register` hace estas operaciones clave:

1. Recupera `model_type`, `metrics`, `main_metric` y `force_register_model` desde Variables de Airflow.
2. Activa el experimento con `mlflow.set_experiment(...)`.
3. Entrena el modelo elegido.
4. Calcula `accuracy` y `precision`.
5. Loggea metricas, parametros, tags y el artifact `model`.

In [ ]:
index = solved_dag_text.index("def train_and_register")
print(solved_dag_text[index:index + 2600])

## Apartado 4: registro condicional del modelo

La solucion busca versiones previas del modelo en el Registry, extrae la mejor `main_metric` histórica y decide:

- Registrar si no existe histórico.
- Registrar si `force_register_model=true`.
- Registrar si la nueva métrica es mayor o igual al mejor histórico.
- Saltar el registro en el resto de casos dejando trazabilidad en tags.

In [ ]:
index = solved_dag_text.index("def get_best_registered_metric")
print(solved_dag_text[index:index + 1800])

## Apartado 5: escenarios que debe cubrir

La solucion queda preparada para validar como mínimo estos escenarios:

- `model_type=random_forest`
- `model_type=logistic_regression`
- `model_type=xgboost`
- `main_metric=accuracy`
- primera ejecucion sin historico
- ejecucion posterior con historico
- `force_register_model=true`
- `force_register_model=false`

In [ ]:
print(textwrap.dedent(
    """
    Validación manual recomendada:
    - Trigger del DAG resuelto desde Airflow
    - Revisar logs de train_and_register
    - Abrir MLflow y comprobar params, metrics, tags y artifact_path=model
    - Comprobar en Models si hubo registro o skip segun la politica de comparacion
    """
))

## Preguntas teórica resueltas

1. ¿Qué ventaja tiene separar `ReaderFactory` de la lógica del DAG? `ReaderFactory` desacopla el origen de datos del DAG y permite cambiar el backend sin rehacer la orquestación.
2. ¿Por qué `force_register_model` puede ser útil? `force_register_model` sirve para pruebas controladas, demos o escenarios en los que interesa versionar una ejecución aunque no mejore la métrica principal.
3. ¿Qué diferencia hay entre loggear un modelo como artifact y registrarlo en el Model Registry? Loggear un modelo como artifact lo deja asociado a un run; registrarlo en el Registry lo convierte en una entidad versionada y consultable por nombre.
4. ¿Por qué el ejercicio compara contra el mejor histórico según `main_metric` y no contra el último? Comparar contra el mejor historico evita promover modelos peores solo por ser más recientes.

## Extensiones opcionales

- Añadir `recall` y `f1` como métricas extras. 
- Persistir la matriz de confusion como artifact.
- Hacer configurable `test_size` desde Variables de Airflow.
- Extender `ReaderFactory` para leer un CSV real y no solo el fallback de iris.